# EvidenceRx -- Evidence-Tier Classifier v2 (PubMedBERT)

**7 classes (index = tier):** 0 off-topic, 1 synthesis, 2 rct, 3 human-observational, 4 animal, 5 in-vitro, 6 review.

### What changed from v1
| | v1 | v2 |
|---|---|---|
| max tokens | 384 | **512** |
| val split | loaded, unused | **used for model selection** |
| seeds | 1 | **3, softmax-ensembled** |
| compared | weak->gold only | **gold-only vs weak->gold, both ensembled** |

**Selection rule:** the winning family is chosen on VAL (208 papers). TEST (147 papers, frozen since round 1) is reported once at the end and never used to choose anything -- that is what keeps it comparable with the round-1 number of 0.803.

Attach the dataset containing `e3_weak.csv`, `e3_gold_train.csv`, `e3_gold_val.csv`, `e3_gold_test.csv`, `e3_predict.csv`; enable GPU + Internet.

In [ ]:
import os, glob, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification, set_seed)
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

_hits = glob.glob('/kaggle/input/**/e3_weak.csv', recursive=True)
assert _hits, 'e3_weak.csv not found under /kaggle/input -- click Add Input'
INPUT_DIR = os.path.dirname(_hits[0])
print('INPUT_DIR =', INPUT_DIR)
if len(_hits) > 1:
    print('WARNING: e3_weak.csv found in more than one input; using the '
          'first. Detach old dataset versions if that is not intended:')
    for h in _hits:
        print('   ', h)

MODEL = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
CLASSES = ['off-topic','synthesis','rct','human-observational',
           'animal','in-vitro','review']   # index == tier
NUM = len(CLASSES)
MAX_LEN = 512
BATCH = 16
WEAK_EPOCHS = 3
GOLD_EPOCHS = 6
LR = 2e-5
FT_LR = 1e-5
SEEDS = [42, 1337, 2024]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| max_len:', MAX_LEN, '| seeds:', SEEDS)

In [ ]:
weak = pd.read_csv(f'{INPUT_DIR}/e3_weak.csv')
gtr  = pd.read_csv(f'{INPUT_DIR}/e3_gold_train.csv')
gva  = pd.read_csv(f'{INPUT_DIR}/e3_gold_val.csv')
gte  = pd.read_csv(f'{INPUT_DIR}/e3_gold_test.csv')
pred = pd.read_csv(f'{INPUT_DIR}/e3_predict.csv')
for d in (weak, gtr, gva, gte, pred):
    d['text'] = d['text'].fillna('').astype(str)
y_val, y_test = gva['label'].values, gte['label'].values
print('weak', len(weak), '| gold tr/va/te', len(gtr), len(gva), len(gte),
      '| predict', len(pred))
print('gold train balance:', gtr['label'].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

class DS(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, padding='max_length',
                        max_length=MAX_LEN, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[i]), dtype=torch.long)
        return item

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=NUM).to(device)

def class_weights(labels):
    counts = np.bincount(labels, minlength=NUM).astype(float)
    w = counts.sum() / (NUM * np.clip(counts, 1, None))
    w[counts == 0] = 1.0
    return torch.tensor(w, dtype=torch.float).to(device)

@torch.no_grad()
def probs_on(model, df):
    # softmax matrix (n, NUM) -- this is the unit we ensemble over
    model.eval()
    out = []
    for b in DataLoader(DS(df['text']), batch_size=BATCH):
        b = {k: v.to(device) for k, v in b.items()}
        out.append(torch.softmax(model(**b).logits, 1).cpu().numpy())
    return np.concatenate(out)

def score(P, y):
    yh = P.argmax(1)
    return accuracy_score(y, yh), f1_score(y, yh, average='macro')

def train(model, df, epochs, lr, tag, seed):
    set_seed(seed)
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(DS(df['text'], df['label']), batch_size=BATCH,
                    shuffle=True, generator=g)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights(df['label'].values))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    for ep in range(epochs):
        model.train()
        tot = 0
        for b in dl:
            y = b.pop('labels').to(device)
            b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad()
            loss = loss_fn(model(**b).logits, y)
            loss.backward()
            opt.step()
            tot += loss.item()
        # VAL only -- test is never looked at during training
        va, vf = score(probs_on(model, gva), y_val)
        print(f'  [{tag}] ep {ep+1}/{epochs} loss={tot/len(dl):.3f} '
              f'val_acc={va:.3f} val_F1={vf:.3f}')
    return model

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ===== Stage 1: WEAK (trained ONCE, reused by every weak->gold seed) =====
print(f'--- Stage 1: weak supervision ({len(weak):,} rule labels) ---')
m_weak = train(new_model(), weak, WEAK_EPOCHS, LR, 'weak', SEEDS[0])
torch.save(m_weak.state_dict(), 'weak_stage.pt')
weak_val = score(probs_on(m_weak, gva), y_val)
weak_test = score(probs_on(m_weak, gte), y_test)
print(f'weak-only  val acc={weak_val[0]:.3f}  test acc={weak_test[0]:.3f}')
del m_weak
torch.cuda.empty_cache()

In [ ]:
# ===== Both families x 3 seeds. Probabilities kept for ensembling. =====
P = {f: {'val': [], 'test': []} for f in ('gold-only', 'weak->gold')}
per_seed = []

for s in SEEDS:
    print(f'\n===== seed {s} =====')
    # --- gold-only: fresh PubMedBERT, gold train split only ---
    print(f'--- gold-only ({len(gtr)} rows) ---')
    m = train(new_model(), gtr, GOLD_EPOCHS, LR, f'gold-only s{s}', s)
    P['gold-only']['val'].append(probs_on(m, gva))
    P['gold-only']['test'].append(probs_on(m, gte))
    per_seed.append(('gold-only', s,
                     *score(P['gold-only']['val'][-1], y_val),
                     *score(P['gold-only']['test'][-1], y_test)))
    del m
    torch.cuda.empty_cache()

    # --- weak->gold: reload the SAME weak checkpoint every seed ---
    print('--- weak->gold ---')
    m = new_model()
    m.load_state_dict(torch.load('weak_stage.pt'))
    m = train(m, gtr, GOLD_EPOCHS, FT_LR, f'weak->gold s{s}', s)
    P['weak->gold']['val'].append(probs_on(m, gva))
    P['weak->gold']['test'].append(probs_on(m, gte))
    per_seed.append(('weak->gold', s,
                     *score(P['weak->gold']['val'][-1], y_val),
                     *score(P['weak->gold']['test'][-1], y_test)))
    del m
    torch.cuda.empty_cache()

seeds_df = pd.DataFrame(per_seed, columns=['family','seed','val_acc',
                        'val_F1','test_acc','test_F1'])
print('\n===== per-seed =====')
print(seeds_df.round(3).to_string(index=False))

In [ ]:
# ===== Seed variance + ensembles. Decision is made on VAL. =====
print('===== seed variance (is the gap real, or noise?) =====')
agg = seeds_df.groupby('family')[['val_acc','test_acc']].agg(['mean','std'])
print(agg.round(3).to_string())

ENS = {f: {sp: np.mean(P[f][sp], axis=0) for sp in ('val','test')}
       for f in P}
rows = []
for f in P:
    sv = seeds_df[seeds_df.family == f]
    rows.append([f, 'single (mean of 3)', sv.val_acc.mean(), sv.val_F1.mean(),
                 sv.test_acc.mean(), sv.test_F1.mean()])
    rows.append([f, 'ENSEMBLE of 3', *score(ENS[f]['val'], y_val),
                 *score(ENS[f]['test'], y_test)])
rows.append(['weak-only', 'single', weak_val[0], weak_val[1],
             weak_test[0], weak_test[1]])
tbl = pd.DataFrame(rows, columns=['family','mode','val_acc','val_F1',
                                  'test_acc','test_F1'])
print('\n===== ABLATION =====')
print(tbl.round(3).to_string(index=False))

cand = tbl[tbl['mode'] == 'ENSEMBLE of 3']
BEST = cand.loc[cand.val_acc.idxmax(), 'family']
print(f'\n>>> WINNER on VAL: {BEST} (ensemble of {len(SEEDS)})')
print('    test accuracy is reported below and was NOT used to choose.')

In [ ]:
# ===== Final report: winning ensemble on the frozen test split =====
yh = ENS[BEST]['test'].argmax(1)
acc, f1 = score(ENS[BEST]['test'], y_test)
print(f'===== FINAL: {BEST} ensemble on TEST (n={len(y_test)}) =====')
print(f'accuracy {acc:.3f}   macro_F1 {f1:.3f}   '
      f'(round-1 baseline: 0.803 / 0.796)\n')
print(classification_report(y_test, yh, labels=list(range(NUM)),
      target_names=CLASSES, digits=3, zero_division=0))
print('confusion (rows=true, cols=pred):')
cm = pd.DataFrame(confusion_matrix(y_test, yh, labels=list(range(NUM))),
                  index=CLASSES, columns=CLASSES)
print(cm)
err = int(cm.values.sum() - np.trace(cm.values))
print(f'\ntotal errors: {err}')
per_cls = (cm.sum(1) - np.diag(cm)).sort_values(ascending=False)
print('errors by TRUE class (where the remaining headroom is):')
for k, v in per_cls.items():
    if v:
        print(f'   {k:22s} {v:3d}  ({100*v/max(err,1):.0f}% of all errors)')

In [ ]:
# ===== Full inference with the winning ensemble =====
acc_probs = np.zeros((len(pred), NUM), dtype=np.float64)
for s in SEEDS:
    print(f'--- refit + inference, seed {s} ---')
    if BEST == 'gold-only':
        m = train(new_model(), gtr, GOLD_EPOCHS, LR, f'refit s{s}', s)
    else:
        m = new_model()
        m.load_state_dict(torch.load('weak_stage.pt'))
        m = train(m, gtr, GOLD_EPOCHS, FT_LR, f'refit s{s}', s)
    acc_probs += probs_on(m, pred)
    del m
    torch.cuda.empty_cache()
Pr = acc_probs / len(SEEDS)

res = pred[['pmid']].copy()
for i, cname in enumerate(CLASSES):
    res[f'p_{i}_{cname}'] = Pr[:, i]
res['pred_tier'] = Pr.argmax(1)
res['pred_class'] = [CLASSES[i] for i in Pr.argmax(1)]
res['confidence'] = Pr.max(1)
res.to_csv('e3_tier_predictions.csv', index=False)
seeds_df.to_csv('e3_seed_results.csv', index=False)
tbl.to_csv('e3_ablation.csv', index=False)
print('\nwrote e3_tier_predictions.csv', res.shape)
print(res['pred_class'].value_counts())
print('mean confidence:', round(res['confidence'].mean(), 3))

## After running
Download **`e3_tier_predictions.csv`**, `e3_ablation.csv` and `e3_seed_results.csv` into the project folder, then locally:
```
python evidence_build_v3.py   # attach tier + off-topic flag
python evidence_aggregate.py  # tier-corrected re-analysis (Phase 6)
```
`e3_ablation.csv` and `e3_seed_results.csv` are the tables for the report: they show gold-only vs weak->gold with seed variance, so the claim rests on means rather than on a 5-paper difference.